# Qwen3-VL-32B — Colab API Server

This notebook **only** serves the model. No extraction, no Drive I/O.

```
Local machine  ──POST /explain──►  ngrok tunnel  ──►  Colab (this notebook)
                ◄── JSON explanation ──────────────────────────────────────
```

### Workflow
1. Run **Cell 1** (GPU check)
2. Run **Cell 2** (install)
3. Run **Cell 3** (load model) — ~5 min download first time
4. Run **Cell 4** (start server) — prints the ngrok URL
5. Copy the URL into `local_qwen_caller.py` → `COLAB_URL`
6. Run `local_qwen_caller.py` on your machine
7. Keep this notebook alive (Cell 4 blocks)


In [ ]:
# ── Cell 1: GPU check ────────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install deps ──────────────────────────────────────────────────────
# Run once, then Runtime → Restart session, then re-run from Cell 1.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'transformers>=4.45.0', 'accelerate', 'qwen-vl-utils[decord]',
    'fastapi', 'uvicorn[standard]', 'pyngrok', 'python-multipart',
])
import transformers
print(f'transformers : {transformers.__version__}')
print('Done. Restart session if this was the first install.')

In [ ]:
# ── Cell 3: Load Qwen3-VL-32B ─────────────────────────────────────────────────
import os, gc, torch
from huggingface_hub import login
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

# ── HF auth ──────────────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get('HF_TOKEN', '')
login(token=HF_TOKEN, add_to_git_credential=False)
print(f'HF authenticated.')

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID = 'Qwen/Qwen3-VL-32B-Instruct'
# MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'   # ← fallback if 32B OOMs

try:
    from transformers import Qwen3VLForConditionalGeneration as QwenVLModel
    print('Using Qwen3VLForConditionalGeneration')
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as QwenVLModel
    print('Falling back to Qwen2VLForConditionalGeneration')

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

print(f'Loading {MODEL_ID} ...')
model = QwenVLModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)
print('Model loaded.')
print(f'GPU allocated : {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'GPU reserved  : {torch.cuda.memory_reserved()/1e9:.1f} GB')

In [ ]:
# ── Cell 4: FastAPI server + ngrok tunnel ─────────────────────────────────────
#
# POST /explain
#   Body (JSON):
#     {
#       "system_prompt": "...",
#       "intro_text":    "...",
#       "frames": [
#         {"label": "Frame from snippet 3 ...", "b64": "<base64 jpeg>"},
#         ...
#       ],
#       "max_new_tokens": 300
#     }
#
# Response (JSON): {"explanation": "...", "raw": "..."}
#
# Keep this cell running — it blocks until you interrupt the kernel.

import json, base64, tempfile, os, threading
from pathlib import Path
from typing import List

import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from pyngrok import ngrok, conf
from qwen_vl_utils import process_vision_info

# ── ngrok auth (free tier — no token needed for basic tunnel) ─────────────────
# If you have an ngrok account, set your authtoken here for stable URLs:
NGROK_TOKEN = ''   # ← optional: paste your ngrok authtoken
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

# ── Request / response schemas ────────────────────────────────────────────────
class FrameItem(BaseModel):
    label: str
    b64:   str          # base64-encoded JPEG

class ExplainRequest(BaseModel):
    system_prompt:  str
    intro_text:     str
    frames:         List[FrameItem]
    max_new_tokens: int = 300

# ── Inference function ────────────────────────────────────────────────────────
def run_inference(req: ExplainRequest) -> dict:
    # Write frames to temp files so qwen_vl_utils can open them
    tmp_paths = []
    try:
        content = [{"type": "text", "text": req.intro_text}]
        for fr in req.frames:
            tmp = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False)
            tmp.write(base64.b64decode(fr.b64))
            tmp.close()
            tmp_paths.append(tmp.name)
            content.append({"type": "text",  "text": fr.label})
            content.append({"type": "image", "image": f"file://{tmp.name}"})

        messages = [
            {"role": "system", "content": req.system_prompt},
            {"role": "user",   "content": content},
        ]

        text_input = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        pvi = process_vision_info(messages)
        image_inputs, video_inputs = pvi[0], pvi[1]

        inputs = processor(
            text=[text_input],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors='pt',
        ).to('cuda')

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, max_new_tokens=req.max_new_tokens
            )

        trimmed = [out[len(inp):] for inp, out in
                   zip(inputs.input_ids, generated_ids)]
        raw = processor.batch_decode(
            trimmed, skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0].strip()

        # Parse JSON or return raw text
        try:
            text = raw
            if text.startswith('```'):
                text = text.split('```')[1]
                if text.startswith('json'):
                    text = text[4:]
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = {"explanation": raw}

        return {"explanation": parsed.get("explanation", raw), "raw": raw}

    finally:
        for p in tmp_paths:
            try:
                os.unlink(p)
            except Exception:
                pass

# ── FastAPI app ───────────────────────────────────────────────────────────────
app = FastAPI(title='Qwen-VL Anomaly Explainer')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID}

@app.post('/explain')
def explain(req: ExplainRequest):
    try:
        result = run_inference(req)
        return JSONResponse(content=result)
    except Exception as e:
        return JSONResponse(status_code=500,
                            content={'error': str(e)})

# ── Start ngrok + uvicorn ─────────────────────────────────────────────────────
PORT = 8000
public_url = ngrok.connect(PORT, bind_tls=True).public_url
print(f'\n{'='*60}')
print(f'  Qwen-VL server live at:')
print(f'  {public_url}')
print(f'  Health check : {public_url}/health')
print(f'  Endpoint     : POST {public_url}/explain')
print(f'{'='*60}')
print(f'  Copy the URL above into local_qwen_caller.py → COLAB_URL')
print(f'  This cell will block until you stop the kernel.')
print(f'{'='*60}\n')

uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='warning')